In [1]:
import pystac
from datetime import datetime
import zarr

In [69]:
s3_endpoint = "https://objects.eodc.eu"
bucket = "68e13833a1624f43ba2cac01376a18af:destine-climate-dt/Austria"
zarr_store = "climate-dt-austria.zarr"
snow_zarr = "IFS-NEMO_sfc_ScenarioMIP_SSP3-7.0_snowDepth.zarr"

store = f"{s3_endpoint}/{bucket}/{zarr_store}"
snow_store = f"{s3_endpoint}/{bucket}/{snow_zarr}"

In [70]:
print(store)
print(snow_store)

https://objects.eodc.eu/68e13833a1624f43ba2cac01376a18af:destine-climate-dt/Austria/climate-dt-austria.zarr
https://objects.eodc.eu/68e13833a1624f43ba2cac01376a18af:destine-climate-dt/Austria/IFS-NEMO_sfc_ScenarioMIP_SSP3-7.0_snowDepth.zarr


In [72]:
extent = pystac.Extent(
    spatial=pystac.SpatialExtent([[4.0,43.0,18.0,50.00]]),
    temporal=pystac.TemporalExtent([[datetime(1990, 1, 1), datetime(2039, 12, 31)]])
)

collection = pystac.Collection(
    id="climatedt-austria",
    title="DestinE Climate Change Adaptation Digital Twin",
    description="The DestinE Climate Change Adaptation Digital Twin (Climate DT) delivers global climate projections and impact-sector information on multi-decadal timescales (1990–2050) at very high spatial resolutions (5–10 km). It integrates state-of-the-art Earth-system models, impact-sector applications, and observations into a unified framework. The Climate DT is the first operational system for global multi-decadal climate projections, running on EuroHPC supercomputing facilities using leading European climate models, including a new generation of global storm-resolving and eddy-rich models. Simulations cover the recent past (from 1990) through possible future climate evolutions to 2050. More information here: https://confluence.ecmwf.int/display/DDCZ/Climate+DT+Phase+1+data+catalogue",
    extent=extent)

collection.stac_extensions = [
    "https://stac-extensions.github.io/zarr/v1.1.0/schema.json",
]

collection.license = "CC-BY-4.0"


# collection.links = [l for l in collection.links if l.rel != "item"]

In [73]:
thumbnail = pystac.Asset(
    href="https://raw.githubusercontent.com/koenifra/thumbnails/main/ClimateDT.png",
    media_type="image/png",
    roles=["thumbnail"],
    title="ClimateDT thumbnail",
    description="Link to thumbnail: https://destine.ecmwf.int/harnessing-the-climate-change-adaptation-digital-twin-for-wind-energy/"
)

collection.add_asset("thumbnail", thumbnail)

In [74]:
bbox = [9.47,46.34,17.22,49.04]
geometry = {
    "type": "Polygon",
    "coordinates": [[
        [bbox[0], bbox[1]],
        [bbox[2], bbox[1]],
        [bbox[2], bbox[3]],
        [bbox[0], bbox[3]],
        [bbox[0], bbox[1]],
    ]]
}


bbox_snow = [4.0,43.0,18.0,50.00]
geometry_snow = {
    "type": "Polygon",
    "coordinates": [[
        [bbox_snow[0], bbox_snow[1]],
        [bbox_snow[2], bbox_snow[1]],
        [bbox_snow[2], bbox_snow[3]],
        [bbox_snow[0], bbox_snow[3]],
        [bbox_snow[0], bbox_snow[1]],
    ]]
}

In [75]:
item_nemo_ScenarioMIP_snow = pystac.Item(
    id="IFS-NEMO-ScenarioMIP-snowDepth",
    geometry=geometry_snow,
    bbox=bbox_snow,
    datetime=None,  # because it's a time range
    properties={
        "title": "Snow Depth IFS-NEMO ScenarioMIP SSP3-7.0",
        "description": "Model: IFS-NEMO \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high \n Parameter: Snow depth",
        "start_datetime": datetime(2020, 1, 1).isoformat() + "Z",
        "end_datetime": datetime(2024, 12, 31).isoformat() + "Z",
    },
)


asset_nemo_ScenarioMIP_snow = pystac.Asset(
    href=snow_store,
    media_type="application/vnd.zarr; version=3",
    roles=["data"],
    title="Snow Depth IFS-NEMO ScenarioMIP SSP3-7.0",
    description="Model: IFS-NEMO \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high \n Parameter: Snow depth",
)

asset_nemo_ScenarioMIP_snow.extra_fields.update({
    "xarray:open_zarr_kwargs": {
        "zarr_format": 3,
        "consolidated": True,
    },

})

item_nemo_ScenarioMIP_snow.add_asset("data", asset_nemo_ScenarioMIP_snow)

# item_nemo_ScenarioMIP.links = [link for link in item_nemo_ScenarioMIP.links if link.href is not None]

In [76]:
item_fesom_sn_cont = pystac.Item(
    id="IFS-FESOM-story-nudging-cont",
    geometry=geometry,
    bbox=bbox,
    datetime=None,  # because it's a time range
    properties={
        "title": "IFS-FESOM story-nudging cont",
        "description": "Model: IFS-FESOM \nActivity: story-nudging \nExperiment: cont \n Resolution: high",
        "start_datetime": datetime(2017, 1, 1).isoformat() + "Z",
        "end_datetime": datetime(2023, 12, 31).isoformat() + "Z",
    },
)


asset_fesom_sn_cont = pystac.Asset(
    href=store,
    media_type="application/vnd.zarr; version=3",
    roles=["data"],
    title="IFS-FESOM story-nudging cont",
    description="Model: IFS-FESOM \nActivity: story-nudging \nExperiment: cont \n Resolution: high",
)

asset_fesom_sn_cont.extra_fields.update({
    "xarray:open_zarr_kwargs": {
        "group": "IFS-FESOM_sfc_story-nudging_cont",
        "zarr_format": 3,
        "consolidated": True,
    },
        "bands": [
            {"name": "2 metre temperature", "description": "Near-surface (usually, 2 meter) air temperature"},
            {"name": "Total precipitation", "description": "Represents the amount of water (rain, snow, etc.) as the depth of liquid water if it were all melted and spread evenly."},

        ],

})

item_fesom_sn_cont.add_asset("data", asset_fesom_sn_cont)

# item_fesom_sn_cont.links = [link for link in item_fesom_sn_cont.links if link.href is not None]


In [77]:
item_fesom_sn_hist = pystac.Item(
    id="IFS-FESOM-story-nudging-hist",
    geometry=geometry,
    bbox=bbox,
    datetime=None,  # because it's a time range
    properties={
        "title": "IFS-FESOM story-nudging hist",
        "description": "Model: IFS-FESOM \nActivity: story-nudging \nExperiment: hist \n Resolution: high",
        "start_datetime": datetime(2017, 1, 1).isoformat() + "Z",
        "end_datetime": datetime(2023, 12, 31).isoformat() + "Z",
    },
)


asset_fesom_sn_hist = pystac.Asset(
    href=store,
    media_type="application/vnd.zarr; version=3",
    roles=["data"],
    title="IFS-FESOM story-nudging hist",
    description="Model: IFS-FESOM \nActivity: story-nudging \nExperiment: hist \n Resolution: high",
)

asset_fesom_sn_hist.extra_fields.update({
    "xarray:open_zarr_kwargs": {
        "group": "IFS-FESOM_sfc_story-nudging_hist",
        "zarr_format": 3,
        "consolidated": True,
    },
        "bands": [
            {"name": "2 metre temperature", "description": "Near-surface (usually, 2 meter) air temperature"},
            {"name": "Total precipitation", "description": "Represents the amount of water (rain, snow, etc.) as the depth of liquid water if it were all melted and spread evenly."},

        ],

})

item_fesom_sn_hist.add_asset("data", asset_fesom_sn_hist)

# item_fesom_sn_hist.links = [link for link in item_fesom_sn_hist.links if link.href is not None]

In [78]:
item_fesom_sn_Tplus2 = pystac.Item(
    id="IFS-FESOM-story-nudging-Tplus2",
    geometry=geometry,
    bbox=bbox,
    datetime=None,  # because it's a time range
    properties={
        "title": "IFS-FESOM story-nudging Tplus2",
        "description": "Model: IFS-FESOM \nActivity: story-nudging \nExperiment: Tplus2.0K \n Resolution: high",
        "start_datetime": datetime(2017, 1, 1).isoformat() + "Z",
        "end_datetime": datetime(2023, 12, 31).isoformat() + "Z",
    },
)


asset_fesom_sn_Tplus2 = pystac.Asset(
    href=store,
    media_type="application/vnd.zarr; version=3",
    roles=["data"],
    title="IFS-FESOM story-nudging Tplus2",
    description="Model: IFS-FESOM \nActivity: story-nudging \nExperiment: Tplus2.0K \n Resolution: high",
)

asset_fesom_sn_Tplus2.extra_fields.update({
    "xarray:open_zarr_kwargs": {
        "group": "IFS-FESOM_sfc_story-nudging_Tplus2.0K",
        "zarr_format": 3,
        "consolidated": True,
    },
        "bands": [
            {"name": "2 metre temperature", "description": "Near-surface (usually, 2 meter) air temperature"},
            {"name": "Total precipitation", "description": "Represents the amount of water (rain, snow, etc.) as the depth of liquid water if it were all melted and spread evenly."},

        ],

})

item_fesom_sn_Tplus2.add_asset("data", asset_fesom_sn_Tplus2)

# item_fesom_sn_Tplus2.links = [link for link in item_fesom_sn_Tplus2.links if link.href is not None]

In [79]:
item_nemo_CMIP6 = pystac.Item(
    id="IFS-NEMO-CMIP6",
    geometry=geometry,
    bbox=bbox,
    datetime=None,  # because it's a time range
    properties={
        "title": "IFS-NEMO CMIP6 hist",
        "description": "Model: IFS-NEMO \nActivity: CMIP6 \nExperiment: hist \n Resolution: high",
        "start_datetime": datetime(1990, 1, 1).isoformat() + "Z",
        "end_datetime": datetime(2001, 12, 31).isoformat() + "Z",
    },
)


asset_nemo_CMIP6 = pystac.Asset(
    href=store,
    media_type="application/vnd.zarr; version=3",
    roles=["data"],
    title="IFS-NEMO CMIP6 hist",
    description="Model: IFS-NEMO \nActivity: CMIP6 \nExperiment: hist \n Resolution: high",
)

asset_nemo_CMIP6.extra_fields.update({
    "xarray:open_zarr_kwargs": {
        "group": "IFS-NEMO_sfc_CMIP6_hist",
        "zarr_format": 3,
        "consolidated": True,
    },
        "bands": [
            {"name": "2 metre temperature", "description": "Near-surface (usually, 2 meter) air temperature"},
            {"name": "Total precipitation", "description": "Represents the amount of water (rain, snow, etc.) as the depth of liquid water if it were all melted and spread evenly."},

        ],

})

item_nemo_CMIP6.add_asset("data", asset_nemo_CMIP6)

# item_nemo_CMIP6.links = [link for link in item_nemo_CMIP6.links if link.href is not None]

In [80]:
item_nemo_HighResMIP = pystac.Item(
    id="IFS-NEMO-HighResMIP",
    geometry=geometry,
    bbox=bbox,
    datetime=None,  # because it's a time range
    properties={
        "title": "IFS-NEMO HighResMIP cont",
        "description": "Model: IFS-NEMO \nActivity: HighResMIP \nExperiment: cont \n Resolution: high",
        "start_datetime": datetime(1990, 1, 1).isoformat() + "Z",
        "end_datetime": datetime(2007, 12, 31).isoformat() + "Z",
    },
)


asset_nemo_HighResMIP = pystac.Asset(
    href=store,
    media_type="application/vnd.zarr; version=3",
    roles=["data"],
    title="IFS-NEMO HighResMIP cont",
    description="Model: IFS-NEMO \nActivity: HighResMIP \nExperiment: cont \n Resolution: high",
)

asset_nemo_HighResMIP.extra_fields.update({
    "xarray:open_zarr_kwargs": {
        "group": "IFS-NEMO_sfc_HighResMIP_cont",
        "zarr_format": 3,
        "consolidated": True,
    },
        "bands": [
            {"name": "2 metre temperature", "description": "Near-surface (usually, 2 meter) air temperature"},
            {"name": "Total precipitation", "description": "Represents the amount of water (rain, snow, etc.) as the depth of liquid water if it were all melted and spread evenly."},

        ],

})

item_nemo_HighResMIP.add_asset("data", asset_nemo_HighResMIP)

# item_nemo_HighResMIP.links = [link for link in item_nemo_HighResMIP.links if link.href is not None]

In [81]:
item_nemo_ScenarioMIP = pystac.Item(
    id="IFS-NEMO-ScenarioMIP",
    geometry=geometry,
    bbox=bbox,
    datetime=None,  # because it's a time range
    properties={
        "title": "IFS-NEMO ScenarioMIP SSP3-7.0",
        "description": "Model: IFS-NEMO \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high",
        "start_datetime": datetime(2020, 1, 1).isoformat() + "Z",
        "end_datetime": datetime(2039, 12, 31).isoformat() + "Z",
    },
)


asset_nemo_ScenarioMIP = pystac.Asset(
    href=store,
    media_type="application/vnd.zarr; version=3",
    roles=["data"],
    title="IFS-NEMO ScenarioMIP SSP3-7.0",
    description="Model: IFS-NEMO \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high",
)

asset_nemo_ScenarioMIP.extra_fields.update({
    "xarray:open_zarr_kwargs": {
        "group": "IFS-NEMO_sfc_ScenarioMIP_SSP3-7.0",
        "zarr_format": 3,
        "consolidated": True,
    },
        "bands": [
            {"name": "2 metre temperature", "description": "Near-surface (usually, 2 meter) air temperature"},
            {"name": "Total precipitation", "description": "Represents the amount of water (rain, snow, etc.) as the depth of liquid water if it were all melted and spread evenly."},

        ],

})

item_nemo_ScenarioMIP.add_asset("data", asset_nemo_ScenarioMIP)

# item_nemo_ScenarioMIP.links = [link for link in item_nemo_ScenarioMIP.links if link.href is not None]

In [82]:
item_icon_CMIP6 = pystac.Item(
    id="ICON-CMIP6",
    geometry=geometry,
    bbox=bbox,
    datetime=None,  # because it's a time range
    properties={
        "title": "ICON CMIP6 hist",
        "description": "Model: ICON \nActivity: CMIP6 \nExperiment: hist \n Resolution: high",
        "start_datetime": datetime(1991, 1, 1).isoformat() + "Z",
        "end_datetime": datetime(2019, 12, 31).isoformat() + "Z",
    },
)


asset_icon_CMIP6 = pystac.Asset(
    href=store,
    media_type="application/vnd.zarr; version=3",
    roles=["data"],
    title="ICON CMIP6 hist",
    description="Model: ICON \nActivity: CMIP6 \nExperiment: hist \n Resolution: high",
)

asset_icon_CMIP6.extra_fields.update({
    "xarray:open_zarr_kwargs": {
        "group": "ICON_sfc_CMIP6_hist",
        "zarr_format": 3,
        "consolidated": True,
    },
        "bands": [
            {"name": "2 metre temperature", "description": "Near-surface (usually, 2 meter) air temperature"},
        ],

})

item_icon_CMIP6.add_asset("data", asset_icon_CMIP6)

# item_nemo_CMIP6.links = [link for link in item_nemo_CMIP6.links if link.href is not None]

In [83]:
item_icon_ScenarioMIP = pystac.Item(
    id="ICON-ScenarioMIP",
    geometry=geometry,
    bbox=bbox,
    datetime=None,  # because it's a time range
    properties={
        "title": "ICON ScenarioMIP SSP3-7.0",
        "description": "Model: ICON \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high",
        "start_datetime": datetime(2020, 1, 1).isoformat() + "Z",
        "end_datetime": datetime(2039, 12, 31).isoformat() + "Z",
    },
)


asset_icon_ScenarioMIP = pystac.Asset(
    href=store,
    media_type="application/vnd.zarr; version=3",
    roles=["data"],
    title="ICON ScenarioMIP SSP3-7.0",
    description="Model: ICON \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high",
)

asset_icon_ScenarioMIP.extra_fields.update({
    "xarray:open_zarr_kwargs": {
        "group": "ICON_sfc_ScenarioMIP_SSP3-7.0",
        "zarr_format": 3,
        "consolidated": True,
    },
        "bands": [
            {"name": "2 metre temperature", "description": "Near-surface (usually, 2 meter) air temperature"},
        ],

})

item_icon_ScenarioMIP.add_asset("data", asset_icon_ScenarioMIP)

# item_nemo_ScenarioMIP.links = [link for link in item_nemo_ScenarioMIP.links if link.href is not None]

In [84]:
collection.add_item(item_fesom_sn_cont)
collection.add_item(item_fesom_sn_hist)
collection.add_item(item_fesom_sn_Tplus2)
collection.add_item(item_nemo_CMIP6)
collection.add_item(item_nemo_HighResMIP)
collection.add_item(item_nemo_ScenarioMIP)
collection.add_item(item_nemo_ScenarioMIP_snow)
collection.add_item(item_icon_CMIP6)
collection.add_item(item_icon_ScenarioMIP)

collection.links = [link for link in collection.links if link.get_href() is not None]

In [85]:
print(collection.to_dict())

{'type': 'Collection', 'id': 'climatedt-austria', 'stac_version': '1.1.0', 'description': 'The DestinE Climate Change Adaptation Digital Twin (Climate DT) delivers global climate projections and impact-sector information on multi-decadal timescales (1990–2050) at very high spatial resolutions (5–10 km). It integrates state-of-the-art Earth-system models, impact-sector applications, and observations into a unified framework. The Climate DT is the first operational system for global multi-decadal climate projections, running on EuroHPC supercomputing facilities using leading European climate models, including a new generation of global storm-resolving and eddy-rich models. Simulations cover the recent past (from 1990) through possible future climate evolutions to 2050. More information here: https://confluence.ecmwf.int/display/DDCZ/Climate+DT+Phase+1+data+catalogue', 'links': [], 'stac_extensions': ['https://stac-extensions.github.io/zarr/v1.1.0/schema.json'], 'title': 'DestinE Climate 

In [ ]:

import requests

collection_url = "https://stac.eodc.eu/ingestion/v1/collections"

In [87]:
# Send a POST request to ingest the STAC Collection
# collection_ingest_response = requests.post(collection_url, json=collection.to_dict(), auth=("", ""))
collection_ingest_response = requests.post(collection_url, json=collection.to_dict(), auth=(user, key))

# Print the response
print("STAC Collection Ingest Response:", collection_ingest_response.text)

STAC Collection Ingest Response: {"id":"climatedt-austria","description":"The DestinE Climate Change Adaptation Digital Twin (Climate DT) delivers global climate projections and impact-sector information on multi-decadal timescales (1990–2050) at very high spatial resolutions (5–10 km). It integrates state-of-the-art Earth-system models, impact-sector applications, and observations into a unified framework. The Climate DT is the first operational system for global multi-decadal climate projections, running on EuroHPC supercomputing facilities using leading European climate models, including a new generation of global storm-resolving and eddy-rich models. Simulations cover the recent past (from 1990) through possible future climate evolutions to 2050. More information here: https://confluence.ecmwf.int/display/DDCZ/Climate+DT+Phase+1+data+catalogue","stac_version":"1.1.0","links":[{"rel":"items","type":"application/geo+json","href":"https://stac.eodc.eu/ingestion/v1/collections/climated

In [88]:
items_url = f"{collection_url}/climatedt-austria/items"

In [89]:
item_nemo_ScenarioMIP_snow.links = [link for link in item_nemo_ScenarioMIP_snow.links if link.get_href() is not None]
response_nemo_ScenarioMIP_snow = requests.post(items_url, json=item_nemo_ScenarioMIP_snow.to_dict(), auth=(user, key))
print("IFS-NEMO-ScenarioMIP-Snow:", response_nemo_ScenarioMIP_snow.status_code, response_nemo_ScenarioMIP_snow.text)

IFS-NEMO-ScenarioMIP-Snow: 201 {"bbox":[4.0,43.0,18.0,50.0],"type":"Feature","geometry":{"type":"Polygon","coordinates":[[[4.0,43.0],[18.0,43.0],[18.0,50.0],[4.0,50.0],[4.0,43.0]]]},"properties":{"title":"Snow Depth IFS-NEMO ScenarioMIP SSP3-7.0","description":"Model: IFS-NEMO \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high \n Parameter: Snow depth","datetime":null,"start_datetime":"2020-01-01T00:00:00Z","end_datetime":"2024-12-31T00:00:00Z"},"id":"IFS-NEMO-ScenarioMIP-snowDepth","stac_version":"1.1.0","assets":{"data":{"href":"https://objects.eodc.eu/68e13833a1624f43ba2cac01376a18af:destine-climate-dt/Austria/IFS-NEMO_sfc_ScenarioMIP_SSP3-7.0_snowDepth.zarr","type":"application/vnd.zarr; version=3","title":"Snow Depth IFS-NEMO ScenarioMIP SSP3-7.0","description":"Model: IFS-NEMO \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high \n Parameter: Snow depth","roles":["data"],"xarray:open_zarr_kwargs":{"zarr_format":3,"consolidated":true}}},"links":[{"rel

In [90]:
item_nemo_CMIP6.links = [link for link in item_nemo_CMIP6.links if link.get_href() is not None]
response_nemo_HighResMIP = requests.post(items_url, json=item_nemo_CMIP6.to_dict(), auth=(user, key))
print("IFS-NEMO-HighResMIP:", response_nemo_HighResMIP.status_code, response_nemo_HighResMIP.text)

IFS-NEMO-HighResMIP: 201 {"bbox":[9.47,46.34,17.22,49.04],"type":"Feature","geometry":{"type":"Polygon","coordinates":[[[9.47,46.34],[17.22,46.34],[17.22,49.04],[9.47,49.04],[9.47,46.34]]]},"properties":{"title":"IFS-NEMO CMIP6 hist","description":"Model: IFS-NEMO \nActivity: CMIP6 \nExperiment: hist \n Resolution: high","datetime":null,"start_datetime":"1990-01-01T00:00:00Z","end_datetime":"2001-12-31T00:00:00Z"},"id":"IFS-NEMO-CMIP6","stac_version":"1.1.0","assets":{"data":{"href":"https://objects.eodc.eu/68e13833a1624f43ba2cac01376a18af:destine-climate-dt/Austria/climate-dt-austria.zarr","type":"application/vnd.zarr; version=3","title":"IFS-NEMO CMIP6 hist","description":"Model: IFS-NEMO \nActivity: CMIP6 \nExperiment: hist \n Resolution: high","roles":["data"],"xarray:open_zarr_kwargs":{"group":"IFS-NEMO_sfc_CMIP6_hist","zarr_format":3,"consolidated":true},"bands":[{"name":"2 metre temperature","description":"Near-surface (usually, 2 meter) air temperature"},{"name":"Total precipit

In [91]:
item_nemo_HighResMIP.links = [link for link in item_nemo_HighResMIP.links if link.get_href() is not None]
response_nemo_HighResMIP = requests.post(items_url, json=item_nemo_HighResMIP.to_dict(), auth=(user, key))
print("IFS-NEMO-HighResMIP:", response_nemo_HighResMIP.status_code, response_nemo_HighResMIP.text)

IFS-NEMO-HighResMIP: 201 {"bbox":[9.47,46.34,17.22,49.04],"type":"Feature","geometry":{"type":"Polygon","coordinates":[[[9.47,46.34],[17.22,46.34],[17.22,49.04],[9.47,49.04],[9.47,46.34]]]},"properties":{"title":"IFS-NEMO HighResMIP cont","description":"Model: IFS-NEMO \nActivity: HighResMIP \nExperiment: cont \n Resolution: high","datetime":null,"start_datetime":"1990-01-01T00:00:00Z","end_datetime":"2007-12-31T00:00:00Z"},"id":"IFS-NEMO-HighResMIP","stac_version":"1.1.0","assets":{"data":{"href":"https://objects.eodc.eu/68e13833a1624f43ba2cac01376a18af:destine-climate-dt/Austria/climate-dt-austria.zarr","type":"application/vnd.zarr; version=3","title":"IFS-NEMO HighResMIP cont","description":"Model: IFS-NEMO \nActivity: HighResMIP \nExperiment: cont \n Resolution: high","roles":["data"],"xarray:open_zarr_kwargs":{"group":"IFS-NEMO_sfc_HighResMIP_cont","zarr_format":3,"consolidated":true},"bands":[{"name":"2 metre temperature","description":"Near-surface (usually, 2 meter) air tempera

In [92]:
item_nemo_ScenarioMIP.links = [link for link in item_nemo_ScenarioMIP.links if link.get_href() is not None]
response_nemo_ScenarioMIP = requests.post(items_url, json=item_nemo_ScenarioMIP.to_dict(), auth=(user, key))
print("IFS-NEMO-ScenarioMIP:", response_nemo_ScenarioMIP.status_code, response_nemo_ScenarioMIP.text)

IFS-NEMO-ScenarioMIP: 201 {"bbox":[9.47,46.34,17.22,49.04],"type":"Feature","geometry":{"type":"Polygon","coordinates":[[[9.47,46.34],[17.22,46.34],[17.22,49.04],[9.47,49.04],[9.47,46.34]]]},"properties":{"title":"IFS-NEMO ScenarioMIP SSP3-7.0","description":"Model: IFS-NEMO \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high","datetime":null,"start_datetime":"2020-01-01T00:00:00Z","end_datetime":"2039-12-31T00:00:00Z"},"id":"IFS-NEMO-ScenarioMIP","stac_version":"1.1.0","assets":{"data":{"href":"https://objects.eodc.eu/68e13833a1624f43ba2cac01376a18af:destine-climate-dt/Austria/climate-dt-austria.zarr","type":"application/vnd.zarr; version=3","title":"IFS-NEMO ScenarioMIP SSP3-7.0","description":"Model: IFS-NEMO \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high","roles":["data"],"xarray:open_zarr_kwargs":{"group":"IFS-NEMO_sfc_ScenarioMIP_SSP3-7.0","zarr_format":3,"consolidated":true},"bands":[{"name":"2 metre temperature","description":"Near-surface (us

In [93]:
item_fesom_sn_cont.links = [link for link in item_fesom_sn_cont.links if link.get_href() is not None]
response_fesom_sn_cont = requests.post(items_url, json=item_fesom_sn_cont.to_dict(), auth=(user, key))
print("IFS-NEMO-ScenarioMIP:", response_nemo_ScenarioMIP.status_code, response_nemo_ScenarioMIP.text)

IFS-NEMO-ScenarioMIP: 201 {"bbox":[9.47,46.34,17.22,49.04],"type":"Feature","geometry":{"type":"Polygon","coordinates":[[[9.47,46.34],[17.22,46.34],[17.22,49.04],[9.47,49.04],[9.47,46.34]]]},"properties":{"title":"IFS-NEMO ScenarioMIP SSP3-7.0","description":"Model: IFS-NEMO \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high","datetime":null,"start_datetime":"2020-01-01T00:00:00Z","end_datetime":"2039-12-31T00:00:00Z"},"id":"IFS-NEMO-ScenarioMIP","stac_version":"1.1.0","assets":{"data":{"href":"https://objects.eodc.eu/68e13833a1624f43ba2cac01376a18af:destine-climate-dt/Austria/climate-dt-austria.zarr","type":"application/vnd.zarr; version=3","title":"IFS-NEMO ScenarioMIP SSP3-7.0","description":"Model: IFS-NEMO \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high","roles":["data"],"xarray:open_zarr_kwargs":{"group":"IFS-NEMO_sfc_ScenarioMIP_SSP3-7.0","zarr_format":3,"consolidated":true},"bands":[{"name":"2 metre temperature","description":"Near-surface (us

In [94]:
item_fesom_sn_hist.links = [link for link in item_fesom_sn_hist.links if link.get_href() is not None]
response_fesom_sn_hist = requests.post(items_url, json=item_fesom_sn_hist.to_dict(), auth=(user, key))
print("IFS-NEMO-ScenarioMIP:", response_nemo_ScenarioMIP.status_code, response_nemo_ScenarioMIP.text)

IFS-NEMO-ScenarioMIP: 201 {"bbox":[9.47,46.34,17.22,49.04],"type":"Feature","geometry":{"type":"Polygon","coordinates":[[[9.47,46.34],[17.22,46.34],[17.22,49.04],[9.47,49.04],[9.47,46.34]]]},"properties":{"title":"IFS-NEMO ScenarioMIP SSP3-7.0","description":"Model: IFS-NEMO \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high","datetime":null,"start_datetime":"2020-01-01T00:00:00Z","end_datetime":"2039-12-31T00:00:00Z"},"id":"IFS-NEMO-ScenarioMIP","stac_version":"1.1.0","assets":{"data":{"href":"https://objects.eodc.eu/68e13833a1624f43ba2cac01376a18af:destine-climate-dt/Austria/climate-dt-austria.zarr","type":"application/vnd.zarr; version=3","title":"IFS-NEMO ScenarioMIP SSP3-7.0","description":"Model: IFS-NEMO \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high","roles":["data"],"xarray:open_zarr_kwargs":{"group":"IFS-NEMO_sfc_ScenarioMIP_SSP3-7.0","zarr_format":3,"consolidated":true},"bands":[{"name":"2 metre temperature","description":"Near-surface (us

In [95]:
item_fesom_sn_Tplus2.links = [link for link in item_fesom_sn_Tplus2.links if link.get_href() is not None]
response_fesom_sn_Tplus2= requests.post(items_url, json=item_fesom_sn_Tplus2.to_dict(), auth=(user, key))
print("IFS-NEMO-ScenarioMIP:", response_nemo_ScenarioMIP.status_code, response_nemo_ScenarioMIP.text)

IFS-NEMO-ScenarioMIP: 201 {"bbox":[9.47,46.34,17.22,49.04],"type":"Feature","geometry":{"type":"Polygon","coordinates":[[[9.47,46.34],[17.22,46.34],[17.22,49.04],[9.47,49.04],[9.47,46.34]]]},"properties":{"title":"IFS-NEMO ScenarioMIP SSP3-7.0","description":"Model: IFS-NEMO \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high","datetime":null,"start_datetime":"2020-01-01T00:00:00Z","end_datetime":"2039-12-31T00:00:00Z"},"id":"IFS-NEMO-ScenarioMIP","stac_version":"1.1.0","assets":{"data":{"href":"https://objects.eodc.eu/68e13833a1624f43ba2cac01376a18af:destine-climate-dt/Austria/climate-dt-austria.zarr","type":"application/vnd.zarr; version=3","title":"IFS-NEMO ScenarioMIP SSP3-7.0","description":"Model: IFS-NEMO \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high","roles":["data"],"xarray:open_zarr_kwargs":{"group":"IFS-NEMO_sfc_ScenarioMIP_SSP3-7.0","zarr_format":3,"consolidated":true},"bands":[{"name":"2 metre temperature","description":"Near-surface (us

In [96]:
item_icon_ScenarioMIP.links = [link for link in item_icon_ScenarioMIP.links if link.get_href() is not None]
response_icon_ScenarioMIP = requests.post(items_url, json=item_icon_ScenarioMIP.to_dict(), auth=(user, key))
print("IFS-ICON-ScenarioMIP:", response_icon_ScenarioMIP.status_code, response_icon_ScenarioMIP.text)

IFS-ICON-ScenarioMIP: 201 {"bbox":[9.47,46.34,17.22,49.04],"type":"Feature","geometry":{"type":"Polygon","coordinates":[[[9.47,46.34],[17.22,46.34],[17.22,49.04],[9.47,49.04],[9.47,46.34]]]},"properties":{"title":"ICON ScenarioMIP SSP3-7.0","description":"Model: ICON \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high","datetime":null,"start_datetime":"2020-01-01T00:00:00Z","end_datetime":"2039-12-31T00:00:00Z"},"id":"ICON-ScenarioMIP","stac_version":"1.1.0","assets":{"data":{"href":"https://objects.eodc.eu/68e13833a1624f43ba2cac01376a18af:destine-climate-dt/Austria/climate-dt-austria.zarr","type":"application/vnd.zarr; version=3","title":"ICON ScenarioMIP SSP3-7.0","description":"Model: ICON \nActivity: ScenarioMIP \nExperiment: SSP3-7.0 \n Resolution: high","roles":["data"],"xarray:open_zarr_kwargs":{"group":"ICON_sfc_ScenarioMIP_SSP3-7.0","zarr_format":3,"consolidated":true},"bands":[{"name":"2 metre temperature","description":"Near-surface (usually, 2 meter) air temp

In [97]:
item_icon_CMIP6.links = [link for link in item_icon_CMIP6.links if link.get_href() is not None]
response_icon_HighResMIP = requests.post(items_url, json=item_icon_CMIP6.to_dict(), auth=(user, key))
print("IFS-ICON-HighResMIP:", response_icon_HighResMIP.status_code, response_icon_HighResMIP.text)

IFS-ICON-HighResMIP: 201 {"bbox":[9.47,46.34,17.22,49.04],"type":"Feature","geometry":{"type":"Polygon","coordinates":[[[9.47,46.34],[17.22,46.34],[17.22,49.04],[9.47,49.04],[9.47,46.34]]]},"properties":{"title":"ICON CMIP6 hist","description":"Model: ICON \nActivity: CMIP6 \nExperiment: hist \n Resolution: high","datetime":null,"start_datetime":"1991-01-01T00:00:00Z","end_datetime":"2019-12-31T00:00:00Z"},"id":"ICON-CMIP6","stac_version":"1.1.0","assets":{"data":{"href":"https://objects.eodc.eu/68e13833a1624f43ba2cac01376a18af:destine-climate-dt/Austria/climate-dt-austria.zarr","type":"application/vnd.zarr; version=3","title":"ICON CMIP6 hist","description":"Model: ICON \nActivity: CMIP6 \nExperiment: hist \n Resolution: high","roles":["data"],"xarray:open_zarr_kwargs":{"group":"ICON_sfc_CMIP6_hist","zarr_format":3,"consolidated":true},"bands":[{"name":"2 metre temperature","description":"Near-surface (usually, 2 meter) air temperature"}]}},"links":[{"rel":"collection","type":"applica

STAC Collection deleted successfully.
